# Credit Card Fraud Detection

**Problem:** predict whether a credit-card transaction is fraudulent.

**Model:** an **Autoencoder** trained only on normal transactions. A fraudulent transaction is rebuilt badly,
so a high **reconstruction error** means fraud. The autoencoder is compared with a **Logistic Regression** model.

| Requirement | Section |
|---|---|
| Data preprocessing | 2–3 |
| Autoencoder | 4 |
| Regularization (Early Stopping) | 4 |
| Hyperparameter tuning | 5 |
| Evaluation metrics | 6 |
| Model comparison | 7 |
| Model saving & loading | 8 |
| Inference pipeline (function) | 9 |

## 1. Setup and data

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve, confusion_matrix,
                             classification_report, precision_score, recall_score, f1_score)

SEED = 42
keras.utils.set_random_seed(SEED)

import kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(path + "/creditcard.csv")
print("Shape:", df.shape)
df.head()

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Exploratory data analysis (EDA)

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["Class"].value_counts().rename({0: "Normal", 1: "Fraud"}))
print(f"Fraud rate: {df['Class'].mean():.4%}")

Missing values: 0
Duplicate rows: 1081
Class
Normal    284315
Fraud        492
Name: count, dtype: int64
Fraud rate: 0.1727%


**Findings:**
- No missing values, but there are duplicate rows → remove them.
- Fraud is only ≈0.17% of transactions → **accuracy is misleading**, so we use precision, recall, F1 and PR-AUC.
- `Amount` is very skewed → use `log(1 + Amount)`.
- `Time` only counts seconds from the start of a 2-day window → drop it.

## 3. Preprocessing

In [ ]:
df = df.drop_duplicates()
FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount"]   # Time is dropped
X = df[FEATURES].copy()
X["Amount"] = np.log1p(X["Amount"])
y = df["Class"].values

# 70% train / 15% validation / 15% test, keeping the same fraud rate in each part
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

# Scaling: fitted on the training set only
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

# The autoencoder learns from NORMAL transactions only
X_train_normal = X_train[y_train == 0]
X_val_normal = X_val[y_val == 0]

print("Train:", X_train.shape, "| frauds:", y_train.sum())
print("Validation:", X_val.shape, "| frauds:", y_val.sum())
print("Test:", X_test.shape, "| frauds:", y_test.sum())
print("Autoencoder training data (normal only):", X_train_normal.shape)

Train: (198608, 29) | frauds: 331
Validation: (42559, 29) | frauds: 71
Test: (42559, 29) | frauds: 71
Autoencoder training data (normal only): (198277, 29)


## 4. Autoencoder model

`29 → 20 → 14 → bottleneck → 14 → 20 → 29`

The encoder compresses each transaction into a small **bottleneck**, and the decoder rebuilds it.
The loss is the mean squared error between the input and its reconstruction.

**Regularization — Early Stopping:** training stops when the validation loss stops improving, and the best
weights are restored. This prevents the model from overfitting the training data.

In [ ]:
def build_autoencoder(bottleneck):
    model = keras.Sequential([
        keras.Input(shape=(29,)),
        layers.Dense(20, activation="relu"),
        layers.Dense(14, activation="relu"),
        layers.Dense(bottleneck),                  # bottleneck
        layers.Dense(14, activation="relu"),
        layers.Dense(20, activation="relu"),
        layers.Dense(29, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, batch_size=8192, verbose=0)), axis=1)

## 5. Hyperparameter tuning

The most important hyperparameter of an autoencoder is the **bottleneck size**: a smaller bottleneck forces
stronger compression, so unusual (fraud) transactions are rebuilt worse. Three sizes are tried and compared
by **PR-AUC on the validation set**.

In [ ]:
results = []
models = {}
for bottleneck in [2, 4, 6]:
    keras.utils.set_random_seed(SEED)
    model = build_autoencoder(bottleneck)
    # Regularization: a NEW Early Stopping for each experiment
    early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    history = model.fit(X_train_normal, X_train_normal,
                        validation_data=(X_val_normal, X_val_normal),
                        epochs=100, batch_size=1024, callbacks=[early_stopping], verbose=0)
    pr_auc = average_precision_score(y_val, reconstruction_error(model, X_val))
    results.append({"bottleneck": bottleneck, "epochs": len(history.history["loss"]), "val_PR_AUC": round(pr_auc, 4)})
    models[bottleneck] = (model, history)

results = pd.DataFrame(results)
results

,bottleneck,epochs,val_PR_AUC
0,2,100,0.1980
1,4,100,0.4501
2,6,100,0.4636


In [ ]:
best_bottleneck = int(results.loc[results["val_PR_AUC"].idxmax(), "bottleneck"])
autoencoder, history = models[best_bottleneck]
print("Best bottleneck:", best_bottleneck)

Best bottleneck: 6


## 6. Evaluation

A transaction is flagged as fraud when its reconstruction error is above a **threshold**.
The threshold is chosen on the validation set (best F1); the test set is used only here.

In [ ]:
def best_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(name, y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"model": name,
            "precision": round(precision_score(y_true, y_pred), 4),
            "recall": round(recall_score(y_true, y_pred), 4),
            "f1": round(f1_score(y_true, y_pred), 4),
            "PR_AUC": round(average_precision_score(y_true, scores), 4),
            "frauds_caught": tp, "frauds_missed": fn, "false_alarms": fp}


threshold = best_threshold(y_val, reconstruction_error(autoencoder, X_val))
test_errors = reconstruction_error(autoencoder, X_test)
print("Threshold:", round(float(threshold), 4))
print(classification_report(y_test, (test_errors >= threshold).astype(int), target_names=["Normal", "Fraud"], digits=4))
print(confusion_matrix(y_test, (test_errors >= threshold).astype(int)))

Threshold: 4.2652
              precision    recall  f1-score   support

      Normal     0.9995    0.9988    0.9992     42488
       Fraud     0.5100    0.7183    0.5965        71

    accuracy                         0.9984     42559
   macro avg     0.7548    0.8586    0.7978     42559
weighted avg     0.9987    0.9984    0.9985     42559

[[42439    49]
 [   20    51]]


## 7. Comparison: Autoencoder vs Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train)
lr_threshold = best_threshold(y_val, logreg.predict_proba(X_val)[:, 1])

comparison = pd.DataFrame([
    evaluate(f"Autoencoder (bottleneck={best_bottleneck})", y_test, test_errors, threshold),
    evaluate("Logistic Regression", y_test, logreg.predict_proba(X_test)[:, 1], lr_threshold),
])
comparison

,model,precision,recall,f1,PR_AUC,frauds_caught,frauds_missed,false_alarms
0,Autoencoder (bottleneck=6),0.5100,0.7183,0.5965,0.5856,51,20,49
1,Logistic Regression,0.7971,0.7746,0.7857,0.6802,55,16,14


## 8. Save and load the model

In [ ]:
autoencoder.save("fraud_autoencoder.h5")
joblib.dump(scaler, "scaler.joblib")
with open("threshold.json", "w") as f:
    json.dump({"threshold": float(threshold)}, f)

loaded_model = keras.models.load_model("fraud_autoencoder.h5", compile=False)
loaded_scaler = joblib.load("scaler.joblib")
loaded_threshold = json.load(open("threshold.json"))["threshold"]

same = np.allclose(reconstruction_error(loaded_model, X_test[:1000]), test_errors[:1000], atol=1e-5)
print("Loaded model gives the same results:", same)

Loaded model gives the same results: True


## 9. Inference pipeline

`predict_transaction` takes one raw transaction (`V1`–`V28` and `Amount`) and returns its reconstruction error
and the fraud decision, using the loaded model, scaler and threshold.

In [ ]:
def predict_transaction(transaction):
    x = pd.DataFrame([transaction])[FEATURES].copy()
    x["Amount"] = np.log1p(x["Amount"])
    x = loaded_scaler.transform(x).astype("float32")
    error = float(reconstruction_error(loaded_model, x)[0])
    return {"reconstruction_error": round(error, 4), "is_fraud": error >= loaded_threshold}


# Try it on 10 random frauds and 10 random normal transactions
frauds = df[df["Class"] == 1].sample(10, random_state=SEED)
normals = df[df["Class"] == 0].sample(10, random_state=SEED)

for name, group in [("Fraud", frauds), ("Normal", normals)]:
    predictions = [predict_transaction(row[FEATURES].to_dict()) for _, row in group.iterrows()]
    flagged = sum(p["is_fraud"] for p in predictions)
    print(f"{name} transactions flagged as fraud: {flagged} of {len(group)}")
    for p in predictions:
        print("   ", p)

Fraud transactions flagged as fraud: 9 of 10
    {'reconstruction_error': 56.9314, 'is_fraud': True}
    {'reconstruction_error': 4.2675, 'is_fraud': True}
    {'reconstruction_error': 57.4099, 'is_fraud': True}
    {'reconstruction_error': 8.7232, 'is_fraud': True}
    {'reconstruction_error': 16.9976, 'is_fraud': True}
    {'reconstruction_error': 63.0704, 'is_fraud': True}
    {'reconstruction_error': 0.4477, 'is_fraud': False}
    {'reconstruction_error': 8.4263, 'is_fraud': True}
    {'reconstruction_error': 65.7452, 'is_fraud': True}
    {'reconstruction_error': 43.2461, 'is_fraud': True}
Normal transactions flagged as fraud: 0 of 10
    {'reconstruction_error': 0.182, 'is_fraud': False}
    {'reconstruction_error': 0.2259, 'is_fraud': False}
    {'reconstruction_error': 0.1492, 'is_fraud': False}
    {'reconstruction_error': 0.2432, 'is_fraud': False}
    {'reconstruction_error': 0.2277, 'is_fraud': False}
    {'reconstruction_error': 0.1871, 'is_fraud': False}
    {'reconstruct

## 10. Results and limitations

**Results:** the comparison table in section 7 shows both models on the test set.
The autoencoder never sees a fraud during training, so it can detect fraud without labelled examples
and can flag new fraud patterns. Logistic Regression needs labelled frauds, which usually gives it fewer false alarms.

**Limitations:**
- The autoencoder only learns "unusual", and some unusual transactions are legitimate (false alarms).
- `V1`–`V28` are anonymised, so the predictions cannot be explained in business terms.
- The data covers only two days in 2013; fraud patterns change over time.